# Bootstrap

> One-call factory that assembles a CapabilityManager + JobQueue + capability bindings — closes the demo-app boilerplate duplication audited across 5 substrate consumers.

In [ ]:
#| default_exp bootstrap

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Tuple, Union

import logging

from cjm_plugin_system.core.manager import CapabilityBinding, CapabilityManager
from cjm_plugin_system.core.queue import JobQueue
from cjm_plugin_system.core.scheduling import ResourceScheduler

_logger = logging.getLogger(__name__)

CapabilitySpec = Union[str, Tuple[str, Optional[Dict[str, Any]]], Mapping[str, Any]]

## Pipeline

Container returned by `create_pipeline`. Holds the assembled `CapabilityManager`,
`JobQueue`, and a `bindings` dict mapping capability names to `CapabilityBinding` views
for the capabilities that loaded successfully. Implements async context-manager
semantics for ergonomic teardown:

```python
async with create_pipeline(capabilities=["whisper", "sys-mon"]) as pipe:
    result = await pipe.bindings["whisper"].execute_async(audio=path)
```

The context-manager exit calls `queue.stop()` + `manager.unload_all()`.

In [ ]:
#| export
@dataclass
class Pipeline:
    """Assembled substrate stack: manager + queue + capability bindings."""
    manager: CapabilityManager  # Discovery + lifecycle
    queue: JobQueue  # Job submission + scheduling
    bindings: Dict[str, CapabilityBinding] = field(default_factory=dict)  # Capability name -> bound view
    
    async def start(self) -> None:
        """Start the job queue's background processor."""
        await self.queue.start()
    
    async def stop(self) -> None:
        """Stop the job queue and unload all capabilities."""
        await self.queue.stop()
        self.manager.unload_all()
    
    async def __aenter__(self) -> "Pipeline":
        await self.start()
        return self
    
    async def __aexit__(self, exc_type, exc, tb) -> None:
        await self.stop()

## Spec normalization

Callers can describe each capability entry in three forms — a bare name (use defaults),
a `(name, config)` tuple, or a mapping with `name` + `config` keys. The internal
helper normalizes all three into a `(name, config)` pair.

In [ ]:
#| export
def _normalize_spec(
    spec: CapabilitySpec  # Raw spec from caller
) -> Tuple[str, Optional[Dict[str, Any]]]:  # (capability_name, optional config)
    """Normalize a capability spec into a `(name, config)` pair.
    
    Accepts a bare string, a `(name, config)` tuple, or a mapping with a 'name'
    key (config under 'config' or the mapping itself minus 'name').
    """
    if isinstance(spec, str):
        return spec, None
    if isinstance(spec, tuple):
        if len(spec) == 1:
            return spec[0], None
        if len(spec) == 2:
            return spec[0], spec[1]
        raise ValueError(f"capability spec tuple must be (name,) or (name, config); got {spec!r}")
    if isinstance(spec, Mapping):
        if "name" not in spec:
            raise ValueError(f"capability spec mapping missing 'name' key: {spec!r}")
        name = spec["name"]
        if "config" in spec:
            return name, spec["config"]
        return name, None
    raise TypeError(f"unsupported capability spec type: {type(spec).__name__}")

## create_pipeline

The canonical assembly factory. Substitutes for the ~70-line boilerplate that
5 audited demos have been repeating. Returns a `Pipeline` whose queue hasn't been
started yet — callers either use it as an async context manager or call
`await pipeline.start()` explicitly.

In [ ]:
#| export
def create_pipeline(
    capabilities: Optional[Iterable[CapabilitySpec]] = None,  # Capabilities to discover + load
    scheduler: Optional[ResourceScheduler] = None,  # Resource policy (default: permissive)
    system_monitor: Optional[str] = None,  # Capability name to register as system monitor
    search_paths: Optional[List[Path]] = None,  # Custom manifest search paths
    queue_kwargs: Optional[Dict[str, Any]] = None,  # Extra kwargs forwarded to JobQueue
    strict: bool = True,  # SG-5 strict config validation on each load
) -> Pipeline:  # Assembled stack ready to start
    """Assemble a CapabilityManager + JobQueue + capability bindings in one call.
    
    Steps performed:
      1. Construct CapabilityManager with the given scheduler + search paths
      2. discover_manifests()
      3. For each spec in `capabilities`: load the capability and create a CapabilityBinding
      4. If `system_monitor` is set, register that capability as the sys-mon
      5. Construct JobQueue (NOT started — caller starts via context manager)
    
    Capabilities that fail to load are logged but do not raise; their entries are
    omitted from `Pipeline.bindings`. Use the returned `Pipeline.manager` to
    inspect which loads succeeded.
    """
    manager = CapabilityManager(scheduler=scheduler, search_paths=search_paths)
    manager.discover_manifests()
    
    bindings: Dict[str, CapabilityBinding] = {}
    for spec in (capabilities or []):
        name, config = _normalize_spec(spec)
        binding = manager.bind(name, default_config=config or {})
        try:
            loaded = binding.load(config=config, strict=strict)
        except Exception as e:
            _logger.error("Failed to load capability %s: %s", name, e)
            continue
        if loaded:
            bindings[name] = binding
        else:
            _logger.warning("Capability %s did not load; omitted from pipeline.bindings", name)
    
    if system_monitor:
        manager.register_system_monitor(system_monitor)
    
    queue = JobQueue(manager=manager, **(queue_kwargs or {}))
    return Pipeline(manager=manager, queue=queue, bindings=bindings)

In [ ]:
# SG-18 regression: spec normalization handles all three accepted forms.
from typing import cast

# Bare string
assert _normalize_spec("whisper") == ("whisper", None)

# Tuple — one-element
assert _normalize_spec(("whisper",)) == ("whisper", None)

# Tuple — name + config
assert _normalize_spec(("whisper", {"model": "large"})) == ("whisper", {"model": "large"})

# Mapping with config
assert _normalize_spec({"name": "whisper", "config": {"model": "tiny"}}) == (
    "whisper", {"model": "tiny"},
)

# Mapping without config
assert _normalize_spec({"name": "whisper"}) == ("whisper", None)

# Bad forms raise
try:
    _normalize_spec(cast(Any, 42))
except TypeError as e:
    print(f"✓ rejected int spec: {e}")

try:
    _normalize_spec(("a", "b", "c"))  # type: ignore
except ValueError as e:
    print(f"✓ rejected 3-tuple: {e}")

try:
    _normalize_spec({"config": {}})  # type: ignore
except ValueError as e:
    print(f"✓ rejected mapping without name: {e}")

# Pipeline dataclass round-trips
from unittest.mock import MagicMock
mgr = MagicMock(spec=CapabilityManager)
q = MagicMock(spec=JobQueue)
pipe = Pipeline(manager=mgr, queue=q)
assert pipe.bindings == {}
assert pipe.manager is mgr
assert pipe.queue is q

print("SG-18 spec normalization + Pipeline shape: PASS")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()